# 02 - Final groups and leak-safe splits

**Goal:** start from `manifest_v1.csv`, optionally merge extra sibling groups using embeddings, then create reproducible grouped train/val/test splits.

Outputs:
- `artifacts/manifest_v2.csv` (final grouped + split manifest)
- `artifacts/split_summary.json`
- `artifacts/preprocess_contract.json` (includes the decided resize policy)

This notebook does **not** modify images.

In [1]:
import json, random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
import torch_directml


In [2]:

PROJECT_ROOT = Path.cwd().parent
ARTIFACTS = PROJECT_ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

dml = torch_directml.device()
print("DirectML device:", torch_directml.device_name(0))


DirectML device: AMD Radeon RX 5700 


### Step 1 - load artifacts from notebooks 00 and 01

In [3]:
manifest_path = ARTIFACTS / "manifest_v1.csv"
class_map_path = ARTIFACTS / "class_map.json"
env_report_path = ARTIFACTS / "env_report.json"

assert manifest_path.exists(), f"Missing {manifest_path}; run notebook 01 first."
assert class_map_path.exists(), f"Missing {class_map_path}; run notebook 01 first."
assert env_report_path.exists(), f"Missing {env_report_path}; run notebook 00 first."

df = pd.read_csv(manifest_path)
class_map = json.loads(class_map_path.read_text(encoding="utf-8"))
env_report = json.loads(env_report_path.read_text(encoding="utf-8"))

required_cols = {"path", "class_id", "label", "group_v1", "w", "h", "conflict_label"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"manifest_v1 is missing columns: {missing}")

df["class_id"] = df["class_id"].astype(int)
df["group_v1"] = df["group_v1"].astype(int)
df["conflict_label"] = df["conflict_label"].astype(bool)

labels = [c["label"] for c in sorted(class_map, key=lambda x: x["id"])]
num_classes = len(labels)

print("rows:", len(df))
print("classes:", num_classes)
print(df.groupby("class_id").size())


rows: 11431
classes: 9
class_id
0    1192
1     985
2     513
3    1162
4    2127
5    1000
6    1909
7     952
8    1591
dtype: int64


In [4]:
# Exclude exact-duplicate conflicts from training splits.
excluded = df[df["conflict_label"]].copy()
work = df[~df["conflict_label"]].copy().reset_index(drop=True)
print("excluded conflicting-label rows:", len(excluded))
print("rows used for split:", len(work))


excluded conflicting-label rows: 0
rows used for split: 11431


### Step 2 - decide resize policy from real image aspect ratios

In [5]:
aspect = work["w"] / work["h"]
near_square = ((aspect >= 0.95) & (aspect <= 1.05)).mean()
resize_policy = "direct_resize" if near_square >= 0.98 else "center_crop_then_resize"

preprocess_contract = {
    "input_size": int(env_report["preprocess"]["input_size"]),
    "color_order": env_report["preprocess"]["color_order"],
    "scale": env_report["preprocess"]["scale"],
    "mean": env_report["preprocess"]["mean"],
    "std": env_report["preprocess"]["std"],
    "resize_policy": resize_policy,
    "near_square_share": float(near_square),
}

(ARTIFACTS / "preprocess_contract.json").write_text(json.dumps(preprocess_contract, indent=2), encoding="utf-8")
print(json.dumps(preprocess_contract, indent=2))


{
  "input_size": 224,
  "color_order": "RGB",
  "scale": "uint8 / 255.0",
  "mean": [
    0.485,
    0.456,
    0.406
  ],
  "std": [
    0.229,
    0.224,
    0.225
  ],
  "resize_policy": "direct_resize",
  "near_square_share": 1.0
}


### Step 3 - embed images and merge extra sibling groups (same class only)

We start from `group_v1` from notebook 01 and only add very conservative merges.
Rule: merge two groups when their centroids are reciprocal top-k neighbors with cosine similarity >= `COSINE_T`.

In [6]:
INPUT_SIZE = int(preprocess_contract["input_size"])
MEAN = preprocess_contract["mean"]
STD = preprocess_contract["std"]

class ImgDataset(Dataset):
    def __init__(self, paths, tfm):
        self.paths = list(paths)
        self.tfm = tfm

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        p = self.paths[idx]
        img = Image.open(p).convert("RGB")
        return self.tfm(img)

tfm = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

paths = work["path"].tolist()
ds = ImgDataset(paths, tfm)
loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=0)

weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1
model = mobilenet_v3_small(weights=weights).to(dml).eval()

emb = []
with torch.no_grad():
    for xb in loader:
        xb = xb.to(dml)
        z = model.features(xb)
        z = model.avgpool(z)
        z = torch.flatten(z, 1)
        z = torch.nn.functional.normalize(z, dim=1)
        emb.append(z.cpu())
emb = torch.cat(emb, dim=0).numpy().astype(np.float32)
print("embedding shape:", emb.shape)


embedding shape: (11431, 576)


In [7]:
COSINE_T = 0.992
TOPK = 3

class DSU:
    def __init__(self, n):
        self.p = list(range(n))

    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]
            a = self.p[a]
        return a

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[rb] = ra

dsu = DSU(len(work))

# Keep all group_v1 links.
for _, g in work.groupby("group_v1"):
    idx = g.index.to_list()
    for j in idx[1:]:
        dsu.union(idx[0], j)

extra_links_by_class = {}

for cid in sorted(work["class_id"].unique()):
    sub = work[work["class_id"] == cid]
    idx_rows = sub.index.to_numpy()

    # Group-level centroids from current group_v1 ids (within this class).
    groups = []
    for gid, g in sub.groupby("group_v1"):
        members = g.index.to_numpy()
        centroid = emb[members].mean(axis=0)
        centroid = centroid / max(np.linalg.norm(centroid), 1e-12)
        groups.append((int(gid), members, centroid))

    if len(groups) <= 1:
        extra_links_by_class[int(cid)] = 0
        continue

    cent = np.stack([g[2] for g in groups]).astype(np.float32)
    sim = cent @ cent.T
    np.fill_diagonal(sim, -2.0)

    k = min(TOPK, sim.shape[1] - 1)
    topk_idx = np.argpartition(-sim, kth=np.arange(k), axis=1)[:, :k]
    topk_sim = np.take_along_axis(sim, topk_idx, axis=1)

    directed = set()
    for i in range(sim.shape[0]):
        for j, s in zip(topk_idx[i], topk_sim[i]):
            if s >= COSINE_T:
                directed.add((i, int(j)))

    reciprocal = set()
    for a, b in directed:
        if (b, a) in directed:
            reciprocal.add((min(a, b), max(a, b)))

    for a, b in reciprocal:
        m0 = groups[a][1][0]
        m1 = groups[b][1][0]
        dsu.union(int(m0), int(m1))

    extra_links_by_class[int(cid)] = len(reciprocal)

work["group_id"] = [dsu.find(i) for i in range(len(work))]
sizes = work.groupby("group_id").size()
print("extra reciprocal links by class:", extra_links_by_class)
print("images:", len(work), "final groups:", len(sizes))
print("largest groups:", sizes.sort_values(ascending=False).head(8).tolist())


extra reciprocal links by class: {0: 0, 1: 0, 2: 0, 3: 0, 4: 1, 5: 0, 6: 0, 7: 0, 8: 0}
images: 11431 final groups: 10591
largest groups: [346, 141, 84, 19, 16, 8, 7, 7]


### Step 4 - grouped stratified split (train/val/test)

Groups are assigned as whole units so no group can leak across splits.

In [8]:
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9

split_order = ["train", "val", "test"]
ratio = {"train": TRAIN_RATIO, "val": VAL_RATIO, "test": TEST_RATIO}

group_split = {}

for cid in sorted(work["class_id"].unique()):
    sub = work[work["class_id"] == cid]
    gsz = sub.groupby("group_id").size().rename("n").reset_index()
    gsz = gsz.sample(frac=1.0, random_state=SEED).sort_values("n", ascending=False).reset_index(drop=True)

    total = int(gsz["n"].sum())
    targets = {s: total * ratio[s] for s in split_order}
    assigned = {s: 0 for s in split_order}
    groups_per_split = {s: [] for s in split_order}

    for _, row in gsz.iterrows():
        gid, n = int(row["group_id"]), int(row["n"])

        deficits = {s: targets[s] - assigned[s] for s in split_order}
        best = sorted(split_order, key=lambda s: (deficits[s], -assigned[s]), reverse=True)[0]
        groups_per_split[best].append(gid)
        assigned[best] += n

    # Ensure each split has at least one group whenever possible.
    non_empty = [s for s in split_order if len(groups_per_split[s]) > 0]
    if len(non_empty) < len(split_order) and len(gsz) >= len(split_order):
        donors = sorted(split_order, key=lambda s: len(groups_per_split[s]), reverse=True)
        for s in split_order:
            if groups_per_split[s]:
                continue
            donor = next((d for d in donors if len(groups_per_split[d]) > 1), None)
            if donor is not None:
                moved = groups_per_split[donor].pop()
                groups_per_split[s].append(moved)

    for s in split_order:
        for gid in groups_per_split[s]:
            group_split[gid] = s

work["split"] = work["group_id"].map(group_split)
if work["split"].isna().any():
    raise RuntimeError("Some rows were not assigned a split.")

cross = work.groupby("group_id")["split"].nunique()
if int((cross > 1).sum()) != 0:
    raise RuntimeError("Leak detected: at least one group appears in multiple splits.")

print(work["split"].value_counts())
print(work.groupby(["split", "class_id"]).size().unstack(fill_value=0))


split
train    8001
val      1716
test     1714
Name: count, dtype: int64
class_id    0    1    2    3     4    5     6    7     8
split                                                   
test      179  148   77  174   319  150   286  143   238
train     834  689  359  814  1489  700  1336  666  1114
val       179  148   77  174   319  150   287  143   239


### Step 5 - save outputs for notebook 03

In [9]:
out_cols = [
    "path", "file", "class_id", "label", "sha1", "bytes", "w", "h", "mode",
    "blur", "brightness", "group_v1", "group_id", "split", "blur_flag"
]
manifest_v2 = work[out_cols].copy()
manifest_v2_path = ARTIFACTS / "manifest_v2.csv"
manifest_v2.to_csv(manifest_v2_path, index=False)

summary = {
    "seed": SEED,
    "rows_total_manifest_v1": int(len(df)),
    "rows_excluded_conflict_label": int(len(excluded)),
    "rows_used": int(len(work)),
    "groups_v1": int(df[~df["conflict_label"]]["group_v1"].nunique()),
    "groups_final": int(work["group_id"].nunique()),
    "cosine_threshold": float(COSINE_T),
    "topk_reciprocal": int(TOPK),
    "extra_links_by_class": {str(k): int(v) for k, v in extra_links_by_class.items()},
    "split_counts": {k: int(v) for k, v in work["split"].value_counts().to_dict().items()},
    "split_by_class": {
        str(cid): {
            s: int(((work["class_id"] == cid) & (work["split"] == s)).sum())
            for s in ["train", "val", "test"]
        }
        for cid in sorted(work["class_id"].unique())
    },
    "resize_policy": resize_policy,
}
summary_path = ARTIFACTS / "split_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("saved", manifest_v2_path)
print("saved", summary_path)
print("saved", ARTIFACTS / "preprocess_contract.json")
print(json.dumps(summary, indent=2))


saved C:\Users\baabyface\Documents\my work\agrilens\artifacts\manifest_v2.csv
saved C:\Users\baabyface\Documents\my work\agrilens\artifacts\split_summary.json
saved C:\Users\baabyface\Documents\my work\agrilens\artifacts\preprocess_contract.json
{
  "seed": 42,
  "rows_total_manifest_v1": 11431,
  "rows_excluded_conflict_label": 0,
  "rows_used": 11431,
  "groups_v1": 10592,
  "groups_final": 10591,
  "cosine_threshold": 0.992,
  "topk_reciprocal": 3,
  "extra_links_by_class": {
    "0": 0,
    "1": 0,
    "2": 0,
    "3": 0,
    "4": 1,
    "5": 0,
    "6": 0,
    "7": 0,
    "8": 0
  },
  "split_counts": {
    "train": 8001,
    "val": 1716,
    "test": 1714
  },
  "split_by_class": {
    "0": {
      "train": 834,
      "val": 179,
      "test": 179
    },
    "1": {
      "train": 689,
      "val": 148,
      "test": 148
    },
    "2": {
      "train": 359,
      "val": 77,
      "test": 77
    },
    "3": {
      "train": 814,
      "val": 174,
      "test": 174
    },
    "4": {